# Variational Autoencoder (VAE)
"Auto-Encoding Variational Bayes" (Kingma & Welling, 2013)

**Idea:** an autoencoder whose code is a **distribution**, not a point. The encoder outputs a Gaussian over latent codes for each image, the decoder reconstructs from a *sample* of it, and a KL term pulls every such Gaussian toward $\mathcal{N}(0, I)$. After training, decoding random draws from $\mathcal{N}(0, I)$ gives new images.

### VAE vs autoencoder vs GAN vs diffusion
| | Autoencoder | VAE | GAN (`gan.ipynb`) | Diffusion (`diffusion.ipynb`) |
|---|---|---|---|---|
| Latent code | a point $z = f(x)$ | a Gaussian $q(z \mid x)$ | noise $z$, no encoder | noise of the same size as the image |
| Loss | reconstruction | reconstruction + **KL to the prior** | adversarial (two networks) | noise-prediction MSE |
| Can sample new data? | no (latent space has holes) | **yes**, $z \sim \mathcal{N}(0, I)$ | yes | yes |
| Has an encoder $x \to z$? | yes | yes | no | no (fixed noising) |
| Likelihood | - | lower bound (ELBO) | none | lower bound |
| Sample quality | - | blurry | sharp | sharp |
| Sampling cost | - | 1 decoder pass | 1 generator pass | hundreds of passes |
| Training | stable | stable | unstable | stable |

Here: an MLP VAE on MNIST, once with a 2-dim latent (so the whole latent space can be plotted) and once with 20 dims (better reconstructions), plus a plain autoencoder to show why the KL term is needed for sampling.

Shape flow: `x (B, 784) -> encoder -> mu, logvar (B, Z) -> z = mu + sigma * eps (B, Z) -> decoder -> logits (B, 784)`

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets
import matplotlib.pyplot as plt

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(42)

## 1. Data - MNIST
60k train / 10k test digits, 28x28, flattened to 784 and kept in $[0, 1]$. Pixel values are treated as Bernoulli probabilities ("how likely is this pixel to be on"), which is what makes binary cross-entropy the reconstruction loss. Labels are used only to color the latent plot.

The whole dataset sits on the GPU; batches are index lookups.

In [ ]:
train_ds = datasets.MNIST("../data", train=True, download=True)
test_ds = datasets.MNIST("../data", train=False, download=True)
Xtr = (train_ds.data.float() / 255).reshape(-1, 784).to(device)   # (60000, 784) in [0, 1]
Xte = (test_ds.data.float() / 255).reshape(-1, 784).to(device)    # (10000, 784)
yte = test_ds.targets
print(Xtr.shape, Xte.shape)

## 2. Model
Generative story: draw a code $z$ from the prior, then draw an image from the decoder.

### Formula: Generative model
$$z \sim p(z) = \mathcal{N}(0, I), \qquad x \sim p_\theta(x \mid z) = \prod_{j=1}^{784} \text{Bernoulli}\big(x_j;\ \sigma(f_\theta(z))_j\big)$$
- $z \in \mathbb{R}^{Z}$: latent code ($Z = 2$ or 20)
- $f_\theta$: decoder network, outputs 784 logits; $\sigma$: sigmoid
- to *train* this we would need $p_\theta(z \mid x)$ (which codes produced this image?), which is intractable

### Formula: Encoder (approximate posterior)
$$q_\phi(z \mid x) = \mathcal{N}\big(z;\ \mu_\phi(x),\ \text{diag}(\sigma^2_\phi(x))\big)$$
- a second network guesses the posterior: for each image, a mean $\mu$ and a variance $\sigma^2$ per latent dim
- the network outputs $\log \sigma^2$ (`logvar`), which can be any real number
- "amortized" inference: one network for all $x$, instead of optimizing a separate $q$ per image

### Formula: Reparameterization trick
$$z = \mu_\phi(x) + \sigma_\phi(x) \odot \epsilon, \qquad \epsilon \sim \mathcal{N}(0, I)$$
- sampling $z \sim q_\phi$ directly is not differentiable with respect to $\phi$
- rewritten this way, the randomness lives in $\epsilon$ (an input), and $z$ is a deterministic, differentiable function of $\mu, \sigma$
- gradients of the reconstruction loss flow through $z$ into the encoder

In [ ]:
class VAE(nn.Module):
    # x (B, 784) -> logits (B, 784), mu (B, Z), logvar (B, Z)
    def __init__(self, latent_dim, hidden=400):
        super().__init__()
        self.enc = nn.Sequential(nn.Linear(784, hidden), nn.ReLU(), nn.Linear(hidden, hidden), nn.ReLU())
        self.mu = nn.Linear(hidden, latent_dim)
        self.logvar = nn.Linear(hidden, latent_dim)
        self.dec = nn.Sequential(nn.Linear(latent_dim, hidden), nn.ReLU(), nn.Linear(hidden, hidden), nn.ReLU(), nn.Linear(hidden, 784))

    def encode(self, x):
        h = self.enc(x)                              # (B, H)
        return self.mu(h), self.logvar(h)            # (B, Z), (B, Z)

    def reparameterize(self, mu, logvar):
        eps = torch.randn_like(mu)                   # (B, Z)
        return mu + (0.5 * logvar).exp() * eps       # sigma = exp(logvar / 2)

    def decode(self, z):
        return self.dec(z)                           # (B, Z) -> (B, 784) logits

    def forward(self, x):
        mu, logvar = self.encode(x)
        return self.decode(self.reparameterize(mu, logvar)), mu, logvar

model = VAE(latent_dim=2).to(device)
print(f"params: {sum(p.numel() for p in model.parameters()) / 1e6:.2f}M")
print([t.shape for t in model(Xtr[:4])])

## 3. The loss: negative ELBO
We want to maximize $\log p_\theta(x)$ but cannot compute it. The **evidence lower bound** is a tractable quantity that is always below it.

### Formula: ELBO
$$\log p_\theta(x) \ \ge\ \underbrace{\mathbb{E}_{q_\phi(z \mid x)}\big[\log p_\theta(x \mid z)\big]}_{\text{reconstruction}} - \underbrace{D_{KL}\big(q_\phi(z \mid x)\ \|\ p(z)\big)}_{\text{regularization}}$$
- first term: decode a sampled $z$ and score the original image; estimated with **one** sample of $\epsilon$ per image
- second term: how far the encoder's Gaussian is from the prior
- the gap between the two sides is $D_{KL}(q_\phi(z \mid x) \,\|\, p_\theta(z \mid x)) \ge 0$: the bound is tight when the encoder matches the true posterior

### Formula: Reconstruction term (Bernoulli decoder)
$$-\log p_\theta(x \mid z) = -\sum_{j=1}^{784} \big[x_j \log \hat x_j + (1 - x_j) \log(1 - \hat x_j)\big]$$
- $\hat x = \sigma(f_\theta(z))$: decoded pixel probabilities
- binary cross-entropy, **summed** over pixels (not averaged), so it is on the same scale as the KL term

### Formula: KL term (closed form for Gaussians)
$$D_{KL}\big(\mathcal{N}(\mu, \sigma^2)\ \|\ \mathcal{N}(0, I)\big) = \frac{1}{2}\sum_{k=1}^{Z} \big(\mu_k^2 + \sigma_k^2 - \log \sigma_k^2 - 1\big)$$
- zero only when $\mu = 0, \sigma = 1$; $\mu^2$ pulls the codes toward the origin, $\sigma^2 - \log \sigma^2$ keeps each code's spread near 1
- without it, the encoder would shrink $\sigma \to 0$ and spread the $\mu$'s far apart: a plain autoencoder
- $\beta$-VAE multiplies this term by $\beta$ ($\beta = 1$ is the VAE)

In [ ]:
def vae_loss(logits, x, mu, logvar, beta=1.0):
    # -> total, reconstruction, KL; each averaged over the batch, summed over pixels / latent dims (nats per image)
    rec = F.binary_cross_entropy_with_logits(logits, x, reduction="none").sum(-1).mean()   # (B, 784) -> scalar
    kl = 0.5 * (mu.pow(2) + logvar.exp() - logvar - 1).sum(-1).mean()                       # (B, Z) -> scalar
    return rec + beta * kl, rec, kl

logits, mu, logvar = model(Xtr[:128])
print([round(v.item(), 2) for v in vae_loss(logits, Xtr[:128], mu, logvar)], "| 784 * log 2 =", round(784 * math.log(2), 2))

## 4. Training
Adam, 20 epochs. The test metric is the negative ELBO in **nats per image** (lower is better): an upper bound on $-\log p_\theta(x)$.

In [ ]:
epochs = 20
batch_size = 128
lr = 1e-3

@torch.no_grad()
def evaluate(model, beta=1.0):
    model.eval()
    tot = rec = kl = 0.0
    for i in range(0, len(Xte), 1000):
        x = Xte[i:i + 1000]
        logits, mu, logvar = model(x)
        l, r, k = vae_loss(logits, x, mu, logvar, beta)
        tot += l.item() * len(x); rec += r.item() * len(x); kl += k.item() * len(x)
    model.train()
    return tot / len(Xte), rec / len(Xte), kl / len(Xte)

def train(model, beta=1.0, log=True):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    history = []
    for epoch in range(1, epochs + 1):
        perm = torch.randperm(len(Xtr), device=device)
        for i in range(0, len(Xtr), batch_size):
            x = Xtr[perm[i:i + batch_size]]                    # (B, 784)
            logits, mu, logvar = model(x)
            loss, _, _ = vae_loss(logits, x, mu, logvar, beta)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        tot, rec, kl = evaluate(model)
        history.append((epoch, tot, rec, kl))
        if log and (epoch % 4 == 0 or epoch == 1):
            print(f"epoch {epoch:2d} | test -ELBO {tot:.2f} = rec {rec:.2f} + KL {kl:.2f}")
    return history

torch.manual_seed(42)
vae2 = VAE(latent_dim=2).to(device)
print("VAE, Z = 2")
hist2 = train(vae2)

torch.manual_seed(42)
vae20 = VAE(latent_dim=20).to(device)
print("VAE, Z = 20")
hist20 = train(vae20)

plt.plot([h[0] for h in hist2], [h[1] for h in hist2], label="Z = 2")
plt.plot([h[0] for h in hist20], [h[1] for h in hist20], label="Z = 20")
plt.xlabel("epoch"); plt.ylabel("test -ELBO (nats / image)"); plt.legend(); plt.show()

## 5. Reconstructions
Top: test images. Middle: reconstruction through 2 numbers ($Z = 2$). Bottom: through 20 numbers. The decoder is fed the mean $\mu$ (no sampling noise).

VAE outputs are **blurry**: where the code is ambiguous, the pixel-wise likelihood is maximized by the average of the plausible images.

In [ ]:
def show(imgs, rows, title=None, cmap="gray"):
    # imgs (n, 784) -> grid with `rows` rows
    n = len(imgs)
    cols = n // rows
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 0.9, rows * 0.9))
    for ax, im in zip(axes.flatten(), imgs):
        ax.imshow(im.reshape(28, 28).cpu(), cmap=cmap); ax.axis("off")
    if title:
        fig.suptitle(title)
    plt.show()

@torch.no_grad()
def reconstruct(model, x):
    model.eval()
    return torch.sigmoid(model.decode(model.encode(x)[0]))   # decode the mean

x = Xte[:12]
show(torch.cat([x, reconstruct(vae2, x), reconstruct(vae20, x)]), rows=3, title="input / Z = 2 / Z = 20")

## 6. The latent space ($Z = 2$)
Left: every test image placed at its code mean $\mu(x)$, colored by digit. No labels were used in training, yet digits form clusters, and the cloud as a whole is packed around the origin at unit scale (the KL term's doing).

Right: the **manifold**. Decode a regular grid of $z$ values. The grid is spaced by Gaussian quantiles, so each cell is equally probable under the prior. Neighboring cells decode to similar digits and there are no "dead" regions.

In [ ]:
@torch.no_grad()
def latent_means(model):
    model.eval()
    return model.encode(Xte)[0].cpu()                     # (10000, Z)

mu2 = latent_means(vae2)
n = 16
normal = torch.distributions.Normal(0.0, 1.0)
q = normal.icdf(torch.linspace(0.03, 0.97, n))             # (n,) quantiles of N(0, 1)
grid = torch.stack(torch.meshgrid(q, q.flip(0), indexing="xy"), dim=-1).reshape(-1, 2).to(device)   # (n*n, 2)
with torch.no_grad():
    imgs = torch.sigmoid(vae2.decode(grid)).reshape(n, n, 28, 28).cpu()
canvas = imgs.permute(0, 2, 1, 3).reshape(n * 28, n * 28)   # (n*28, n*28) one big image

fig, ax = plt.subplots(1, 2, figsize=(14, 6.5))
sc = ax[0].scatter(mu2[:, 0], mu2[:, 1], c=yte, cmap="tab10", s=2)
ax[0].set_xlabel("$z_1$"); ax[0].set_ylabel("$z_2$"); ax[0].set_aspect("equal"); plt.colorbar(sc, ax=ax[0], ticks=range(10))
ax[1].imshow(canvas, cmap="gray", extent=[q[0], q[-1], q[0], q[-1]]); ax[1].set_xlabel("$z_1$"); ax[1].set_ylabel("$z_2$")
plt.show()
print("mean of code means:", mu2.mean(0).tolist(), "| std:", mu2.std(0).tolist())

## 7. Sampling, and why the KL term matters
New digits: draw $z \sim \mathcal{N}(0, I)$ and decode. This only works if the codes of real images actually fill the region where the prior puts its mass.

For contrast, a plain **autoencoder**: the same network trained with $\beta = 0$ (no KL term; the encoder is free to put codes anywhere and shrink $\sigma$ to nothing). It reconstructs better, but its codes do not follow $\mathcal{N}(0, I)$, so decoding prior samples gives garbage.

In [ ]:
torch.manual_seed(42)
ae20 = VAE(latent_dim=20).to(device)
train(ae20, beta=0.0, log=False)

mu_vae, mu_ae = latent_means(vae20), latent_means(ae20)
for name, m, mu in [("VAE (beta = 1)", vae20, mu_vae), ("AE  (beta = 0)", ae20, mu_ae)]:
    _, rec, kl = evaluate(m)
    print(f"{name} | test rec {rec:6.2f} | KL to prior {kl:8.2f} | std of code means across the test set {mu.std(0).mean().item():.2f}")

z = torch.randn(24, 20, device=device)                    # the same prior samples for both models
with torch.no_grad():
    show(torch.sigmoid(vae20.decode(z)), rows=2, title="VAE: decode z ~ N(0, I)")
    show(torch.sigmoid(ae20.decode(z)), rows=2, title="autoencoder: decode z ~ N(0, I)")

## 8. Interpolation
Encode two test digits, walk in a straight line between their codes, decode each step. A smooth latent space turns one digit into the other through plausible intermediate shapes rather than a cross-fade of pixels.

In [ ]:
@torch.no_grad()
def interpolate(model, a, b, steps=12):
    za, zb = model.encode(a[None])[0], model.encode(b[None])[0]          # (1, Z) each
    w = torch.linspace(0, 1, steps, device=device)[:, None]              # (steps, 1)
    return torch.sigmoid(model.decode((1 - w) * za + w * zb))            # (steps, 784)

pairs = [(0, 1), (3, 4), (5, 6)]
show(torch.cat([interpolate(vae20, Xte[a], Xte[b]) for a, b in pairs]), rows=3, title="latent interpolation (Z = 20)")

## Notes
- The two loss terms pull in opposite directions: reconstruction wants every image to have its own far-away, sharply located code; KL wants all codes to be the same standard Gaussian. The balance is what gives a latent space that is both informative and sampleable. $\beta > 1$ ($\beta$-VAE) trades reconstruction for more disentangled latents.
- With $Z = 20$ the model does not use all dimensions equally: some stay at $\mu \approx 0, \sigma \approx 1$ (KL $\approx 0$) and carry no information. In the extreme, with a very strong decoder, all of them do (**posterior collapse**).
- Blurriness comes from the factorized pixel likelihood plus the Gaussian posterior, not from the idea itself. Fixes: hierarchical VAEs (NVAE), discrete latents (**VQ-VAE**), or adding perceptual and adversarial losses.
- That last variant is the VAE that matters most today: **latent diffusion** (Stable Diffusion, `dit.ipynb`) first compresses images $8\times$ per side with a KL- or VQ-regularized autoencoder, then runs diffusion in that latent space. The VAE gives the compact space; the diffusion model gives the sharp samples.
- The reparameterization trick is general: any time a sample must be differentiated through (policy noise in RL, Gumbel-softmax for discrete choices), the same rewrite applies.
- The reported number is a bound. A tighter estimate of $\log p(x)$ uses many samples of $z$ per image (importance-weighted bound, IWAE).